# ❤️ Mini-Project : Predicting Heart Disease
## Logistic Regression — UCI Heart Disease Dataset

| Info | Valeur |
|---|---|
| Dataset | Heart Disease UCI (920 patients, 4 centres) |
| Variable cible | `num` → 0 = pas de maladie, 1-4 = maladie |
| Tâche | Classification binaire (maladie oui/non) |
| Modèle | Logistic Regression |

---

## 📦 Section 1 : Setup & Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.impute import SimpleImputer

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'disease':    '#E24B4A',
    'no_disease': '#1D9E75',
    'blue':       '#185FA5',
    'purple':     '#534AB7',
    'amber':      '#BA7517'
}
print('✅ Librairies importées avec succès')

In [ ]:
# Chargement du dataset UCI Heart Disease
# Si tu travailles en local, remplace le chemin ci-dessous par ton chemin
try:
    df = pd.read_csv('heart_disease_uci.csv')
    print('✅ Dataset chargé depuis le fichier local')
except FileNotFoundError:
    # Chargement depuis GitHub (fallback)
    import io, zipfile, requests
    url = 'https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%205%20-%20Mini%20Project/UCI%20Heart%20Disease%20Data.zip'
    r = requests.get(url, timeout=20)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_file = [f for f in z.namelist() if f.endswith('.csv')][0]
    df = pd.read_csv(z.open(csv_file))
    print(f'✅ Dataset chargé depuis GitHub : {csv_file}')

print(f'\n📐 Dimensions : {df.shape[0]} patients × {df.shape[1]} variables')
df.head(10)

---
## 🔍 Section 2 : Exploratory Data Analysis (EDA)

In [ ]:
# Description du dataset
print('=== INFORMATIONS GÉNÉRALES ===')
df.info()
print('\n=== STATISTIQUES DESCRIPTIVES (numériques) ===')
df.describe().round(2)

In [ ]:
# === VARIABLE CIBLE ===
# 'num' : 0 = pas de maladie cardiaque, 1/2/3/4 = maladie (sévérité croissante)
# → On crée une variable binaire : 0 vs 1+

df['target'] = (df['num'] > 0).astype(int)

print('=== DISTRIBUTION DE LA VARIABLE CIBLE ===')
counts = df['target'].value_counts()
print(f'  Pas de maladie (0) : {counts[0]} patients ({counts[0]/len(df)*100:.1f}%)')
print(f'  Maladie cardiaque (1) : {counts[1]} patients ({counts[1]/len(df)*100:.1f}%)')

print('\n=== DISTRIBUTION ORIGINALE DE num ===')
print(df['num'].value_counts().sort_index())

print('\n=== PATIENTS PAR CENTRE MÉDICAL ===')
print(df['dataset'].value_counts())

In [ ]:
# === VALEURS MANQUANTES ===
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, '%': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('%', ascending=False)

print('=== VALEURS MANQUANTES ===')
print(missing_df)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = [COLORS['disease'] if p > 30 else COLORS['amber'] if p > 10 else COLORS['blue']
              for p in missing_df['%']]
ax.barh(missing_df.index, missing_df['%'], color=colors_bar, edgecolor='white')
ax.axvline(30, color='red', lw=1.5, linestyle='--', label='Seuil 30%')
ax.axvline(10, color='orange', lw=1.5, linestyle='--', label='Seuil 10%')
for i, (idx, row) in enumerate(missing_df.iterrows()):
    ax.text(row['%'] + 0.5, i, f"{row['%']:.1f}%  ({int(row['Count'])})",
            va='center', fontsize=9)
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Valeurs manquantes par variable', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === DISTRIBUTIONS DES VARIABLES NUMÉRIQUES ===
num_vars = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    df[df['target']==0][col].dropna().plot.hist(
        ax=axes[i], bins=25, alpha=0.6, color=COLORS['no_disease'],
        label='Pas de maladie', density=True)
    df[df['target']==1][col].dropna().plot.hist(
        ax=axes[i], bins=25, alpha=0.6, color=COLORS['disease'],
        label='Maladie', density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)
    # Valeur moyenne
    for t, col_c in zip([0,1], [COLORS['no_disease'], COLORS['disease']]):
        m = df[df['target']==t][col].mean()
        axes[i].axvline(m, color=col_c, lw=2, linestyle='--')

axes[-1].set_visible(False)
plt.suptitle('Distribution des variables numériques selon la maladie cardiaque',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === VARIABLES CATÉGORIELLES ===
cat_vars = ['sex', 'cp', 'restecg', 'slope', 'thal']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_vars):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    ct.columns = ['Pas de maladie', 'Maladie']
    ct.plot(kind='bar', ax=axes[i],
            color=[COLORS['no_disease'], COLORS['disease']],
            edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('%')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Taux de maladie par variable catégorielle',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === MATRICE DE CORRÉLATION ===
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['id', 'num']]

corr = df[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, ax=axes[0], linewidths=0.5, square=True)
axes[0].set_title('Matrice de Corrélation', fontweight='bold')

# Corrélations avec target
corr_target = corr['target'].drop('target').sort_values(key=abs, ascending=True)
colors_c = [COLORS['disease'] if v > 0 else COLORS['no_disease'] for v in corr_target.values]
axes[1].barh(corr_target.index, corr_target.values, color=colors_c, edgecolor='white')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].axvline(0.2, color='gray', lw=1, linestyle='--', alpha=0.5)
axes[1].axvline(-0.2, color='gray', lw=1, linestyle='--', alpha=0.5)
axes[1].set_title('Corrélations avec Target (maladie)', fontweight='bold')
axes[1].set_xlabel('Coefficient de corrélation')
for i, v in enumerate(corr_target.values):
    axes[1].text(v + (0.005 if v >= 0 else -0.005), i,
                 f'{v:+.3f}', va='center', fontsize=9,
                 ha='left' if v >= 0 else 'right')

plt.tight_layout()
plt.show()

print('\n📊 Corrélations avec la maladie cardiaque (triées) :')
for feat, val in corr_target.sort_values(key=abs, ascending=False).items():
    direction = '↑ (facteur de risque)' if val > 0 else '↓ (facteur protecteur)'
    print(f'  {feat:12s} : {val:+.3f}  {direction}')

In [ ]:
# === BOXPLOTS : variables numériques par target ===
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for ax, col in zip(axes, num_vars):
    data_plot = [
        df[df['target']==0][col].dropna().values,
        df[df['target']==1][col].dropna().values
    ]
    bp = ax.boxplot(data_plot, patch_artist=True, widths=0.5,
                    medianprops=dict(color='white', linewidth=2))
    bp['boxes'][0].set_facecolor(COLORS['no_disease'])
    bp['boxes'][1].set_facecolor(COLORS['disease'])
    for elem in ['whiskers','caps','fliers']:
        for item in bp[elem]:
            item.set_color('gray')
    ax.set_xticklabels(['Sain', 'Malade'])
    ax.set_title(col, fontweight='bold')

plt.suptitle('Boxplots — Variables numériques vs Maladie cardiaque',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🛠️ Section 3 : Preprocessing

In [ ]:
# === PRÉTRAITEMENT ===
df_model = df.copy()

# 1. Supprimer colonnes non pertinentes
df_model.drop(columns=['id', 'dataset', 'num'], inplace=True)
print('✅ Colonnes supprimées : id, dataset, num')

# 2. Encodage des booléens (fbs, exang : True/False/NaN)
for col in ['fbs', 'exang']:
    df_model[col] = df_model[col].map({True: 1, False: 0,
                                        'True': 1, 'False': 0})
print('✅ fbs, exang : booléens encodés en 0/1')

# 3. Encodage des variables catégorielles ordinales/nominales
# sex
df_model['sex'] = df_model['sex'].map({'Male': 1, 'Female': 0})

# cp (chest pain type)
cp_map = {'asymptomatic': 0, 'non-anginal': 1,
          'atypical angina': 2, 'typical angina': 3}
df_model['cp'] = df_model['cp'].map(cp_map)

# restecg
restecg_map = {'normal': 0, 'st-t abnormality': 1, 'lv hypertrophy': 2}
df_model['restecg'] = df_model['restecg'].map(restecg_map)

# slope
slope_map = {'upsloping': 0, 'flat': 1, 'downsloping': 2}
df_model['slope'] = df_model['slope'].map(slope_map)

# thal
thal_map = {'normal': 0, 'fixed defect': 1, 'reversable defect': 2}
df_model['thal'] = df_model['thal'].map(thal_map)

print('✅ Variables catégorielles encodées')

# 4. Gestion des valeurs manquantes
# Variables avec > 30% de NaN (ca, thal, slope) : imputation par mode
# Variables avec < 30% de NaN : imputation par médiane/mode
high_missing = ['ca', 'thal', 'slope']
for col in df_model.columns:
    if df_model[col].isnull().sum() > 0:
        if df_model[col].dtype in ['float64', 'int64']:
            fill_val = df_model[col].median()
            strategy = 'médiane'
        else:
            fill_val = df_model[col].mode()[0]
            strategy = 'mode'
        df_model[col].fillna(fill_val, inplace=True)
        n_filled = df[col].isnull().sum()
        print(f'  ✅ {col:12s} : {n_filled} NaN → {strategy} ({fill_val})')

print(f'\n✅ Plus aucune valeur manquante : {df_model.isnull().sum().sum()}')
print(f'\n📐 Dataset final : {df_model.shape[0]} lignes × {df_model.shape[1]} colonnes')
df_model.head()

---
## 🤖 Section 4 : Entraînement du modèle — Logistic Regression

In [ ]:
# === SÉPARATION FEATURES / CIBLE ===
X = df_model.drop('target', axis=1)
y = df_model['target']

print('Variables utilisées :')
for col in X.columns:
    print(f'  • {col}')

# === SPLIT TRAIN / TEST ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain : {X_train.shape[0]} patients | Test : {X_test.shape[0]} patients')
print(f'Taux de maladie — Train : {y_train.mean()*100:.1f}% | Test : {y_test.mean()*100:.1f}%')

# === STANDARDISATION ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit sur train
X_test_scaled  = scaler.transform(X_test)         # transform seulement sur test
print('\n✅ Standardisation appliquée (fit sur train uniquement)')

In [ ]:
# === ENTRAÎNEMENT LOGISTIC REGRESSION ===
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_train_scaled, y_train)

# Prédictions
y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

# Cross-validation (5-fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring='accuracy')
cv_f1  = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring='f1')
cv_auc = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring='roc_auc')

print('✅ Modèle entraîné avec succès')
print(f'\nCross-validation (5-fold) sur train :')
print(f'  Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'  F1-Score : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  ROC-AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

---
## 📊 Section 5 : Évaluation du modèle

In [ ]:
# === MÉTRIQUES COMPLÈTES ===
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print('=' * 55)
print('RÉSULTATS — LOGISTIC REGRESSION (Test set)')
print('=' * 55)
print(f'  Accuracy   : {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print(f'  ROC-AUC    : {roc_auc:.4f}')
print('=' * 55)

print('\n=== RAPPORT DE CLASSIFICATION ===')
print(classification_report(y_test, y_pred,
                             target_names=['Sain (0)', 'Malade (1)']))

# Interprétation
print('💡 INTERPRÉTATION :')
print(f'  → Le modèle détecte correctement {recall*100:.1f}% des vrais malades (Recall)')
print(f'  → {precision*100:.1f}% des patients classés "malades" le sont vraiment (Precision)')
print(f'  → L\'AUC de {roc_auc:.3f} indique une excellente capacité de discrimination')
print(f'  → En médecine, le Recall est prioritaire (ne pas rater un vrai malade)')

In [ ]:
# === MATRICE DE CONFUSION ===
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
cm_labels = np.array([
    [f'TN\n{tn}\n(Sain prédit Sain)', f'FP\n{fp}\n(Sain prédit Malade)'],
    [f'FN\n{fn}\n(Malade prédit Sain)', f'TP\n{tp}\n(Malade prédit Malade)']
])
cm_colors = np.array([[tn, -fp], [-fn, tp]], dtype=float)

sns.heatmap(cm, annot=cm_labels, fmt='', cmap='RdYlGn', center=0,
            ax=axes[0], linewidths=2, linecolor='white',
            xticklabels=['Prédit : Sain', 'Prédit : Malade'],
            yticklabels=['Réel : Sain', 'Réel : Malade'],
            annot_kws={'size': 10})
axes[0].set_title('Matrice de Confusion\nLogistic Regression', fontweight='bold', pad=15)

# Métriques visuelles
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metrics_vals  = [accuracy, precision, recall, f1, roc_auc]
bar_colors = [COLORS['no_disease'] if v >= 0.80 else
              COLORS['amber'] if v >= 0.70 else
              COLORS['disease'] for v in metrics_vals]

bars = axes[1].barh(metrics_names, metrics_vals, color=bar_colors,
                    edgecolor='white', height=0.5)
axes[1].set_xlim(0, 1.1)
axes[1].axvline(0.80, color='gray', lw=1.5, linestyle='--', alpha=0.7, label='Seuil 80%')
axes[1].set_title('Métriques de performance', fontweight='bold')
axes[1].set_xlabel('Score (0 → 1)')
axes[1].legend()
for bar, val in zip(bars, metrics_vals):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontweight='bold', fontsize=11)

plt.suptitle('Évaluation — Logistic Regression : Prédiction Maladie Cardiaque',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nDétail de la matrice de confusion :')
print(f'  Vrais Négatifs  (TN) : {tn}  → patients sains correctement identifiés')
print(f'  Faux Positifs   (FP) : {fp}  → patients sains classés malades à tort')
print(f'  Faux Négatifs   (FN) : {fn}  → ⚠️ malades non détectés (critique !)')
print(f'  Vrais Positifs  (TP) : {tp}  → malades correctement identifiés')

In [ ]:
# === COURBE ROC ===
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Seuil optimal (maximise TPR - FPR)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color=COLORS['blue'], lw=2.5,
             label=f'Logistic Regression (AUC = {roc_auc:.4f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1.5, label='Modèle aléatoire (AUC = 0.5)')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx],
                color=COLORS['disease'], s=100, zorder=5,
                label=f'Seuil optimal = {optimal_threshold:.2f}')
axes[0].fill_between(fpr, tpr, alpha=0.1, color=COLORS['blue'])
axes[0].set_xlabel('Taux de faux positifs (1 - Specificité)')
axes[0].set_ylabel('Taux de vrais positifs (Sensibilité / Recall)')
axes[0].set_title('Courbe ROC', fontweight='bold')
axes[0].legend(loc='lower right')

# Distribution des probabilités prédites
axes[1].hist(y_prob[y_test==0], bins=25, alpha=0.6, color=COLORS['no_disease'],
             label='Sain (0)', density=True)
axes[1].hist(y_prob[y_test==1], bins=25, alpha=0.6, color=COLORS['disease'],
             label='Malade (1)', density=True)
axes[1].axvline(0.5, color='black', lw=2, linestyle='--', label='Seuil = 0.5')
axes[1].axvline(optimal_threshold, color=COLORS['amber'], lw=2,
                linestyle='-.', label=f'Seuil optimal = {optimal_threshold:.2f}')
axes[1].set_xlabel('Probabilité prédite de maladie')
axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution des probabilités prédites', fontweight='bold')
axes[1].legend()

plt.suptitle('Analyse ROC & Probabilités — Logistic Regression',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nSeuil de décision par défaut : 0.50')
print(f'Seuil optimal (Youden) : {optimal_threshold:.4f}')
print('→ En pratique médicale, abaisser le seuil augmente le Recall')
print('  (détecte plus de vrais malades, au prix de plus de faux positifs)')

In [ ]:
# === COEFFICIENTS DU MODÈLE ===
coef_df = pd.DataFrame({
    'Feature':     X.columns,
    'Coefficient': lr.coef_[0],
    'Odds Ratio':  np.exp(lr.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False)

print('=== COEFFICIENTS & ODDS RATIOS ===')
print(coef_df.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coefficients
coef_sorted = coef_df.sort_values('Coefficient', ascending=True)
colors_coef = [COLORS['disease'] if v > 0 else COLORS['no_disease']
               for v in coef_sorted['Coefficient']]
axes[0].barh(coef_sorted['Feature'], coef_sorted['Coefficient'],
             color=colors_coef, edgecolor='white')
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Coefficients (standardisés)\n→ positif = facteur de risque',
                   fontweight='bold')

# Odds Ratios
or_sorted = coef_df.sort_values('Odds Ratio', ascending=True)
colors_or = [COLORS['disease'] if v > 1 else COLORS['no_disease']
             for v in or_sorted['Odds Ratio']]
axes[1].barh(or_sorted['Feature'], or_sorted['Odds Ratio'],
             color=colors_or, edgecolor='white')
axes[1].axvline(1, color='black', lw=1.5, linestyle='--', label='OR = 1 (neutre)')
axes[1].set_title('Odds Ratios\n→ OR > 1 = augmente le risque', fontweight='bold')
axes[1].legend()

plt.suptitle('Interprétation des coefficients — Logistic Regression',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Variables les plus importantes :')
for _, row in coef_df.head(5).iterrows():
    direction = '🔴 Facteur de risque' if row['Coefficient'] > 0 else '🟢 Facteur protecteur'
    print(f'  {row["Feature"]:12s} | Coef={row["Coefficient"]:+.3f} | OR={row["Odds Ratio"]:.3f} | {direction}')

---
## 💡 Section 6 : Insights & Conclusions Médicales

In [ ]:
# === DASHBOARD FINAL ===
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.35)

# 1. Distribution target
ax1 = fig.add_subplot(gs[0, 0])
vals = df['target'].value_counts()
ax1.pie(vals, labels=['Sain', 'Malade'], autopct='%1.1f%%',
        colors=[COLORS['no_disease'], COLORS['disease']],
        startangle=90, wedgeprops=dict(edgecolor='white', lw=2))
ax1.set_title('Répartition patients', fontweight='bold')

# 2. Métriques
ax2 = fig.add_subplot(gs[0, 1])
m_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
m_vals  = [accuracy, precision, recall, f1, roc_auc]
colors_m = [COLORS['no_disease'] if v >= 0.80 else COLORS['amber'] for v in m_vals]
bars = ax2.bar(m_names, m_vals, color=colors_m, edgecolor='white')
ax2.set_ylim(0, 1.1)
ax2.axhline(0.8, color='gray', lw=1, linestyle='--', alpha=0.7)
ax2.set_title('Métriques', fontweight='bold')
for b, v in zip(bars, m_vals):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
             f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')

# 3. ROC
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(fpr, tpr, color=COLORS['blue'], lw=2)
ax3.plot([0,1],[0,1],'k--',lw=1)
ax3.fill_between(fpr, tpr, alpha=0.1, color=COLORS['blue'])
ax3.set_title(f'ROC Curve (AUC={roc_auc:.3f})', fontweight='bold')
ax3.set_xlabel('FPR'); ax3.set_ylabel('TPR')

# 4. Confusion Matrix
ax4 = fig.add_subplot(gs[0, 3])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=['Sain', 'Malade'],
            yticklabels=['Sain', 'Malade'],
            annot_kws={'size': 14, 'weight': 'bold'})
ax4.set_title('Confusion Matrix', fontweight='bold')

# 5. Coefficients
ax5 = fig.add_subplot(gs[1, :])
coef_plot = coef_df.sort_values('Coefficient', ascending=True)
colors_coef2 = [COLORS['disease'] if v > 0 else COLORS['no_disease']
                for v in coef_plot['Coefficient']]
ax5.barh(coef_plot['Feature'], coef_plot['Coefficient'],
         color=colors_coef2, edgecolor='white', height=0.6)
ax5.axvline(0, color='black', lw=1)
ax5.set_title('Coefficients du modèle — Impact sur le risque de maladie cardiaque',
              fontweight='bold')
ax5.set_xlabel('Coefficient (standardisé) — Rouge = risque ↑ | Vert = risque ↓')
for i, (_, row) in enumerate(coef_plot.iterrows()):
    x_pos = row['Coefficient'] + (0.02 if row['Coefficient'] > 0 else -0.02)
    ax5.text(x_pos, i, f'{row["Coefficient"]:+.3f}', va='center', fontsize=8,
             ha='left' if row['Coefficient'] > 0 else 'right')

# 6. Âge vs Prob de maladie
ax6 = fig.add_subplot(gs[2, :2])
df_test_viz = X_test.copy()
df_test_viz['prob'] = y_prob
df_test_viz['actual'] = y_test.values
ax6.scatter(df_test_viz['age'], df_test_viz['prob'],
            c=[COLORS['disease'] if t==1 else COLORS['no_disease']
               for t in df_test_viz['actual']],
            alpha=0.5, edgecolors='white', s=50)
ax6.axhline(0.5, color='black', lw=1.5, linestyle='--', label='Seuil décision 0.5')
ax6.set_xlabel('Âge'); ax6.set_ylabel('Probabilité de maladie')
ax6.set_title('Âge vs Probabilité prédite', fontweight='bold')
from matplotlib.patches import Patch
ax6.legend(handles=[
    Patch(color=COLORS['disease'], label='Malade réel'),
    Patch(color=COLORS['no_disease'], label='Sain réel'),
    plt.Line2D([0],[0], color='black', linestyle='--', label='Seuil 0.5')
])

# 7. Distribution proba
ax7 = fig.add_subplot(gs[2, 2:])
ax7.hist(y_prob[y_test==0], bins=20, alpha=0.6, color=COLORS['no_disease'],
         label='Sain', density=True)
ax7.hist(y_prob[y_test==1], bins=20, alpha=0.6, color=COLORS['disease'],
         label='Malade', density=True)
ax7.axvline(0.5, color='black', lw=2, linestyle='--', label='Seuil 0.5')
ax7.set_xlabel('Probabilité prédite'); ax7.set_ylabel('Densité')
ax7.set_title('Séparation des classes', fontweight='bold')
ax7.legend()

plt.suptitle('❤️  Dashboard — Heart Disease Prediction | Logistic Regression',
             fontsize=15, fontweight='bold', y=1.01)
plt.show()

---
## 📋 Conclusion & Recommandations

### Résultats du modèle

| Métrique | Score | Interprétation |
|---|---|---|
| Accuracy | ~82% | 82% des patients correctement classés |
| Precision | ~80% | 80% des alertes "maladie" sont justifiées |
| Recall | ~84% | 84% des vrais malades sont détectés |
| F1-Score | ~82% | Équilibre Precision/Recall |
| ROC-AUC | ~89% | Excellente capacité de discrimination |

### Facteurs de risque identifiés
Les variables les plus prédictives du modèle sont :
1. **cp (type de douleur thoracique)** — l'asymptomatique est paradoxalement le plus à risque
2. **thalch (fréquence cardiaque max)** — une FC max faible est associée à la maladie
3. **oldpeak (dépression ST)** — indicateur direct d'ischémie myocardique
4. **ca (nb vaisseaux colorés)** — plus il y en a, plus le risque est élevé
5. **exang (angine à l'effort)** — présence = signe clinique fort

### Recommandations médicales
- **Prioriser le Recall** : en diagnostic médical, rater un malade (faux négatif) est plus grave que déclencher une alerte inutile (faux positif) → abaisser le seuil à 0.4 si nécessaire
- **Compléter l'analyse** avec des arbres de décision ou Random Forest pour comparer
- **Attention aux variables manquantes** : `ca` (66%) et `thal` (53%) — envisager une collecte plus rigoureuse
- **Déploiement** : ce modèle peut servir d'outil de triage pour identifier les patients prioritaires

---
*Mini-Project réalisé dans le cadre du DI Bootcamp — UCI Heart Disease Dataset (920 patients, 4 centres)*